### SEQUENCE

In [1]:
import pandas as pd
import numpy as np

- stock s&p500 상위 30개 종목
- crypto 상위 10개 종목

- 수익률 기준 데이터 불러오기 
$$
R_t = \frac{P_t}{P_{t-1}} - 1
$$ 

In [2]:
return_panel = pd.read_csv("returns.csv", index_col=0, parse_dates=True)

In [3]:
print(return_panel.shape)
print(return_panel.index.min(), return_panel.index.max())
return_panel.head() 

(1826, 40)
2021-04-09 00:00:00 2026-04-08 00:00:00


,AAPL,ABBV,AMD,AMZN,AVGO,BAC,BRK-B,CAT,COST,CVX,...,ADA-USD,BNB-USD,BTC-USD,DOGE-USD,ETH-USD,SOL-USD,TRX-USD,USDC-USD,USDT-USD,XRP-USD
Date,,,,,,,,,,,,,,,,,,,,,
2021-04-09,0.020252,0.013572,-0.007079,0.022096,-0.000803,0.007305,0.009487,0.001171,0.005509,-0.000971,...,-0.013036,0.084059,-0.001354,0.003579,-0.007883,0.027532,-0.057983,0.001253,0.000931,-0.030319
2021-04-10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.012644,0.042769,0.026581,0.035033,0.030806,-0.033727,0.086679,-0.000901,-0.000628,0.346362
2021-04-11,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.039365,0.111785,0.006886,0.169222,0.010166,0.040627,-0.028813,0.002278,0.001707,-0.010103
2021-04-12,-0.013233,0.006602,-0.050507,0.002132,-0.002927,0.001750,0.007218,0.000780,0.004405,-0.011076,...,0.038992,0.139587,-0.005174,-0.052003,-0.008483,0.020773,0.054719,-0.002492,-0.002235,0.078797
2021-04-13,0.024306,-0.000277,0.020489,0.006099,0.002667,-0.018472,-0.003098,-0.006842,0.001096,0.004519,...,0.072682,-0.082068,0.060274,0.320460,0.074712,-0.037758,0.133671,-0.000915,-0.000662,0.222292


In [4]:
stock_tickers = [
    'NVDA',   # 엔비디아
    'AAPL',   # 애플
    'MSFT',   # 마이크로소프트
    'AMZN',   # 아마존
    'GOOGL',  # 알파벳 A
    'META',   # 메타 플랫폼스
    'BRK-B',  # 버크셔 해서웨이
    'LLY',    # 일라이 릴리
    'AVGO',   # 브로드컴
    'TSLA',   # 테슬라
    'WMT',    # 월마트
    'JPM',    # JP모건 체이스
    'UNH',    # 유나이티드헬스 그룹
    'XOM',    # 엑슨모빌
    'V',      # 비자
    'MA',     # 마스터카드
    'JNJ',    # 존슨앤드존슨
    'PG',     # 프록터 앤 갬블
    'ORCL',   # 오라클
    'HD',     # 홈디포
    'COST',   # 코스트코
    'ABBV',   # 애비비
    'BAC',    # 뱅크오브아메리카
    'NFLX',   # 넷플릭스
    'CVX',    # 쉐브론
    'AMD',    # AMD
    'KO',     # 코카콜라
    'CAT',    # 캐터필러
    'PLTR',   # 팔란티어
    'MU'      # 마이크론 테크놀로지
]

crypto_tickers = [
    'BTC-USD',   # 1. 비트코인 
    'ETH-USD',   # 2. 이더리움 
    'USDT-USD',  # 3. 테더 
    'BNB-USD',   # 4. 바이낸스 코인
    'XRP-USD',   # 5. 리플 
    'USDC-USD',  # 6. 유에스디 코인
    'SOL-USD',   # 7. 솔라나 
    'TRX-USD',   # 8. 트론
    'DOGE-USD',  # 9. 도지코인 
    'ADA-USD'    # 10. 카르다노
]

### study period
- `train set`과 `trading set`으로 구성
- 논문기준 study period = 1000days (train: 750days / trading: 250days)
- 데이터의 크기를 고려하여 이번 실험은 500(400/100)으로 진행

In [5]:
#rolling block의 첫 시작은 0일부터
train_size = 608-152
test_size = 152
start_idx = 0 

In [6]:
train_returns = return_panel.iloc[start_idx : start_idx + train_size]
test_returns = return_panel.iloc[start_idx + train_size : start_idx + train_size + test_size]

`train`, `trading`이 잘 나뉘어졌는지 확인
- `train`, `trading`내부에서도 정규화를 위해 `stock` `crypto`데이터를 분리

In [7]:
print("train shape:", train_returns.shape)
print("test shape:", test_returns.shape)

print("train dates:", train_returns.index.min(), "->", train_returns.index.max())
print("test dates:", test_returns.index.min(), "->", test_returns.index.max())

train shape: (456, 40)
test shape: (152, 40)
train dates: 2021-04-09 00:00:00 -> 2022-07-08 00:00:00
test dates: 2022-07-09 00:00:00 -> 2022-12-07 00:00:00


In [8]:
train_stock = train_returns[stock_tickers]
train_crypto = train_returns[crypto_tickers]

test_stock = test_returns[stock_tickers]
test_crypto = test_returns[crypto_tickers]

In [9]:
print(f"train_stock: {train_stock.shape}")
print(train_stock.iloc[:,:5].head(3))
print()

print(f"train_crypto: {train_crypto.shape}")
print(train_crypto.iloc[:,:5].head(3))
print()

print(f"test_stock: {test_stock.shape}")
print(test_stock.iloc[:,:5].head(3))
print()

print(f"test_crypto: {test_crypto.shape}")
print(test_crypto.iloc[:,:5].head(3))
print()


train_stock: (456, 30)
                NVDA      AAPL      MSFT      AMZN     GOOGL
Date                                                        
2021-04-09  0.005797  0.020252  0.010266  0.022096  0.008994
2021-04-10  0.000000  0.000000  0.000000  0.000000  0.000000
2021-04-11  0.000000  0.000000  0.000000  0.000000  0.000000

train_crypto: (456, 10)
             BTC-USD   ETH-USD  USDT-USD   BNB-USD   XRP-USD
Date                                                        
2021-04-09 -0.001354 -0.007883  0.000931  0.084059 -0.030319
2021-04-10  0.026581  0.030806 -0.000628  0.042769  0.346362
2021-04-11  0.006886  0.010166  0.001707  0.111785 -0.010103

test_stock: (152, 30)
                NVDA      AAPL      MSFT      AMZN     GOOGL
Date                                                        
2022-07-09  0.000000  0.000000  0.000000  0.000000  0.000000
2022-07-10  0.000000  0.000000  0.000000  0.000000  0.000000
2022-07-11 -0.043314 -0.014758 -0.011769 -0.032803 -0.030808

test_crypto: 

### 표준화
- stock, crypto 표준화를 위해 각각의 $\mu$, $\sigma$ 를 구한다.

In [10]:
mu_stock = train_stock.stack().mean()
sigma_stock = train_stock.stack().std()
print(f"{mu_stock}, {sigma_stock}")

mu_crypto = train_crypto.stack().mean()
sigma_crypto = train_crypto.stack().std()
print(f"{mu_crypto}, {sigma_crypto}")

0.00010559045253855879, 0.018386160510832937
0.0005503681590510623, 0.05531728028102276


$$
\tilde{R}_{t}^{m,s}
=
\frac{R_{t}^{m,s}-\mu_{\mathrm{train}}^{m}}{\sigma_{\mathrm{train}}^{m}}.
$$

- 표준화 진행
- `test_set`도 `train_set` $\mu$, $\sigma$ 기준으로 표준화

In [11]:
train_stock_std = (train_stock - mu_stock) / sigma_stock
test_stock_std = (test_stock - mu_stock) / sigma_stock

train_crypto_std = (train_crypto - mu_crypto) / sigma_crypto
test_crypto_std = (test_crypto - mu_crypto) / sigma_crypto

- 각각 표준화 된 `stock data`와 `crypto data`를 다시 하나의 universe로 병합

In [12]:
train_std = pd.concat([train_stock_std, train_crypto_std], axis=1)
test_std = pd.concat([test_stock_std, test_crypto_std], axis=1)

In [13]:
all_tickers = stock_tickers + crypto_tickers

train_std = train_std[all_tickers]
test_std = test_std[all_tickers]

In [14]:
print(f"train_std: {train_std.shape}")
print(train_std.iloc[:,:5].head(3))
print(train_std.isnull().sum().max())
print()

print(f"test_std: {test_std.shape}")
print(test_std.iloc[:,:5].head(3))
print(test_std.isnull().sum().max())
print()

train_std: (456, 40)
                NVDA      AAPL      MSFT      AMZN     GOOGL
Date                                                        
2021-04-09  0.309554  1.095727  0.552631  1.196010  0.483426
2021-04-10 -0.005743 -0.005743 -0.005743 -0.005743 -0.005743
2021-04-11 -0.005743 -0.005743 -0.005743 -0.005743 -0.005743
0

test_std: (152, 40)
                NVDA      AAPL      MSFT      AMZN     GOOGL
Date                                                        
2022-07-09 -0.005743 -0.005743 -0.005743 -0.005743 -0.005743
2022-07-10 -0.005743 -0.005743 -0.005743 -0.005743 -0.005743
2022-07-11 -2.361510 -0.808395 -0.645817 -1.789829 -1.681328
0



### 표준화 점검

`stock`, `crypto`의 `train_set`
- $\mu$ 는 0과 근사
- $\sigma$ 는 1.0

In [15]:
print("train stock mean:", train_std[stock_tickers].stack().mean())
print("train stock std :", train_std[stock_tickers].stack().std())

print("train crypto mean:", train_std[crypto_tickers].stack().mean())
print("train crypto std :", train_std[crypto_tickers].stack().std())

train stock mean: 4.15522067695965e-18
train stock std : 0.9999999999999999
train crypto mean: 0.0
train crypto std : 1.0


- `target_panel` 생성 해당 날짜에 맞춰야할 '정답'이 들어있는 데이터 셋
- `median` 보다 크면 1, 작으면 0을 반환
- 다음날 정답을 맞춰야하므로 `shift(-1)`로 맞추고 마지막 행은 .dropna()로 nan값 삭제

In [16]:
returns_target = return_panel.shift(-1)

In [17]:
returns_median = returns_target.median(axis=1)
print(returns_median)

Date
2021-04-09    0.000000
2021-04-10    0.000000
2021-04-11    0.000189
2021-04-12    0.001407
2021-04-13   -0.000274
                ...   
2026-04-04    0.000000
2026-04-05   -0.000365
2026-04-06    0.003424
2026-04-07    0.009003
2026-04-08         NaN
Length: 1826, dtype: float64


In [18]:
target_panel = returns_target.ge(returns_median, axis=0).astype(int)
target_panel = target_panel.iloc[:-1]

In [19]:
print(f"target_panel: {target_panel.shape}")
print(target_panel.iloc[:,:5].tail(3))
print(target_panel.isnull().sum().max())
print()

target_panel: (1825, 40)
            AAPL  ABBV  AMD  AMZN  AVGO
Date                                   
2026-04-05     1     0    1     1     0
2026-04-06     0     0    1     1     1
2026-04-07     1     1    1     1     1
0

